<a href="https://colab.research.google.com/github/DanieleBaiocco/nlp_first_assignment/blob/main/nlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
from urllib import request
from zipfile import ZipFile
import numpy as np
import pandas as pd
import sys
from tensorflow.keras.preprocessing.sequence import pad_sequences


In [2]:

print(f"Current work directory: {os.getcwd()}")
dataset_folder = os.path.join(os.getcwd(), "Datasets")

if not os.path.exists(dataset_folder):
    os.makedirs(dataset_folder)

url = "https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages/corpora/dependency_treebank.zip"
dataset_path = os.path.join(dataset_folder, "Corpora.zip")
print(dataset_path)

def download_dataset(download_path: str, url: str):
    if not os.path.exists(download_path):
        print("Downloading dataset...")
        request.urlretrieve(url, download_path)
        print("Download complete!")

def extract_dataset(download_path: str, extract_path: str):
    print("Extracting dataset... (it may take a while...)")
    with ZipFile(download_path, 'r') as zObject:
        zObject.extractall(
		path=extract_path)
    print("Extraction completed!")

download_dataset(dataset_path, url)
extract_dataset(dataset_path, dataset_folder)

Current work directory: /content
/content/Datasets/Corpora.zip
Extracting dataset... (it may take a while...)
Extraction completed!


In [3]:
def doc_classification(doc_number ):
  if doc_number <= 100:
    split = 'train'
  elif 100 < doc_number <=150:
    split= 'validation'
  else: split = 'test'
  return split 
  
  # 
def preprocessing(word):
    wordtr = ""
    splitted_word = word.split('-')
    if len(splitted_word) > 1:
      wordtr = splitted_word[1]
    else: wordtr = splitted_word[0]
    #try:
    #    float(wordtr)
    #    wordtr = '36'
    #except ValueError:
    #   pass
    return wordtr.lower()

def build_df():
  df_rows = []
  dataset_folder = os.path.join(os.getcwd(), "Datasets", "dependency_treebank")
  list_dir = os.listdir(dataset_folder) 
  list_dir.sort()

  doc_number = 1
  for dir in list_dir:
    file_path = os.path.join(dataset_folder, dir)               
    if os.path.isfile(file_path):
      with open(file_path, mode = 'r', encoding = 'utf-8') as text_file:
        critical_line = 0
        df_row = []
        lines = text_file.readlines()
        for count, line in enumerate(lines):
          splitted_line = line.split()
          if len(splitted_line) != 3 or count + 1 >= len(lines):
              df_rows.append(df_row)
              df_row = []
              critical_line = count + 1
              if count + 1 >= len(lines):
                doc_number += 1
          else:
            if count == critical_line:
              df_row.append(doc_classification(doc_number))
            df_row.append((preprocessing(splitted_line[0]), splitted_line[1]))  
                  

  return df_rows


df = build_df()
#for col in df:
#  df.iloc[:, col].replace([None], 0, inplace=True)




In [4]:
def split_in(dataset, split):
  dataset_first_frac = []
  dataset_last_frac = []
  for sentence in dataset:
    if(sentence[0] == split):
      dataset_first_frac.append(sentence[1:])
    else:
      dataset_last_frac.append(sentence) 
  return dataset_first_frac, dataset_last_frac
    

In [5]:
def split_train_val_test(df):
  df_train, rest = split_in(df, 'train')
  df_val, rest = split_in(rest, 'validation')
  df_test, rest = split_in(rest, 'test')

  return df_train, df_val, df_test
df_train, df_val, df_test = split_train_val_test(df)


In [6]:
import gensim
import pickle
import gensim.downloader as gloader
def load_embedding_model(loading_dir_path, embedding_dimension = 100):
    loading_model_path = os.path.join(loading_dir_path, "model.bin")
    #if not os.path.exists(loading_dir_path):
    download_path = "glove-wiki-gigaword-{}".format(embedding_dimension)
    emb_model = gloader.load(download_path)
    #os.makedirs(loading_dir_path)
    #print(emb_model['ape'])
    #emb_model.save(loading_model_path)
    #else: 
    #  file_model = open(loading_model_path, 'rb')
    #  emb_model = pickle.load(file_model)
    return emb_model 

embedding_model = load_embedding_model(os.path.join(os.getcwd(), "embedding_model"), embedding_dimension=100)
embedding_model.vocab['ape']

[==================================================] 100.0% 128.1/128.1MB downloaded


In [7]:
def check_OOV_terms(embedding_model,
                    dataset):
  
    embedding_vocabulary = set(embedding_model.vocab.keys())
    word_listing = set()
    for sentence in dataset:
      for couple in sentence:
        word, _ = couple
        word_listing.add(word)
    oov = word_listing.difference(embedding_vocabulary)

    oov_percentage = float(len(list(oov))) * 100 / len(word_listing)
    print(f"Total OOV terms: {len(list(oov))} ({oov_percentage:.2f}%)")
    print(list(oov))
    return list(oov)

In [8]:
import copy
def add_OOV_terms(vocabulary, oov, embedding_dim):
  voc = copy.deepcopy(vocabulary)
  for word in oov:
    voc[word] = np.random.uniform(-1, 1, size=embedding_dim)
  print(f"Generated embeddings for {len(oov)} OOV words.")
  print(type(voc))
  return voc

In [9]:
v1 = embedding_model
oov_terms_train = check_OOV_terms(v1, df_train)
v2 = add_OOV_terms(v1, oov_terms_train, embedding_dim=100)
oov_terms_val = check_OOV_terms(v2, df_val)
v3 = add_OOV_terms(v2, oov_terms_val, embedding_dim=100)
oov_terms_test = check_OOV_terms(v3, df_test)
v4 = add_OOV_terms(v3, oov_terms_test, embedding_dim=100)

Total OOV terms: 157 (2.22%)
['', 'delwin', 'amphobiles', 'solaia', 'ensrud', 'samnick', '456.64', 'nesb', 'boorse', 'jalaalwalikraam', 'superpremiums', 'chilver', 'ratners', 'micronite', 'lezovich', 'rapanelli', '497.34', '16,072', 'univest', 'pathlogy', 'synergistics', '3057', 'muscolina', '374.19', '449.04', 'colonsville', 'subskill', '278.7', '62.625', '737.5', '415.6', '95,142', '2645.90', 'jerritts', 'kalipharma', 'ctbs', 'alurralde', 'derel', 'gingl', 'nagymaros', '16.125', '143.93', 'landonne', 'pramual', 'weisfield', 'monchecourt', 'autions', '1.457', 'hummerstone', 'bellringers', 'ntg', 'hallwood', 'wtd', '143.08', '4.898', 'subminimum', 'chinchon', 'bridgestone\\/firestone', 'satrum', '127.03', 'northy', 'c.j.b.', 'besuboru', '4,393,237', 'greenmailer', 'ghkm', '5.276', 'aslacton', 'twindam', 'chafic', 'yeargin', 'tarwhine', "creator's", 'makato', 'vitulli', '1\\/2', '2,303,328', '142.85', 'norwick', 'sanderoff', 'purepac', 'wheeland', '271,124', 'vinken', '14,821', 'molecul

In [10]:
v1['pierre']

array([-0.42799  , -0.47924  , -0.4601   ,  0.067726 ,  0.71207  ,
        0.24324  ,  1.2301   , -0.33719  , -0.265    , -0.84255  ,
        0.016761 ,  0.61517  , -0.30112  , -0.20452  ,  0.75068  ,
        0.24531  , -0.13681  , -0.16167  , -0.66652  ,  0.22497  ,
       -0.3381   ,  0.91272  ,  0.75554  , -0.31635  ,  0.2401   ,
       -1.1912   ,  0.023368 , -0.71069  ,  0.43384  ,  0.044357 ,
       -0.55625  , -0.097796 , -0.58412  , -0.0070207, -0.92978  ,
       -0.085017 ,  0.79975  ,  0.77496  , -0.62987  , -0.57001  ,
        0.48871  ,  0.47946  ,  0.25216  , -0.31682  , -0.032576 ,
        0.18729  , -1.3344   , -0.83018  , -0.68021  ,  0.19414  ,
        0.28167  ,  0.15842  , -0.93762  ,  0.42522  , -0.44457  ,
       -1.1482   , -0.68946  ,  1.0738   ,  0.3171   , -0.11838  ,
        0.03871  , -0.31168  ,  0.31538  , -0.067675 ,  0.45268  ,
       -0.63845  , -0.04799  ,  0.1773   , -0.43704  ,  0.76607  ,
        0.09944  , -0.80918  , -1.3846   , -0.92872  ,  0.2606

In [11]:
from tensorflow.keras.utils import pad_sequences
import itertools

def build_embeddings_df(vocabulary, df, embedding_dimension, max_l):
  df_rows = []
  ys = []
  for sentence in df:
    embeddings_row = np.array([vocabulary[word] for word, _ in sentence])
    df_rows.append(
         np.vstack(
             (
             embeddings_row,
             np.zeros((max_l - len(sentence), embedding_dimension)),
             )
         )
    )
    y = [target for _, target in sentence]
    ys.append(y)

  
  return np.array(df_rows), np.array(ys)


max_len_sentence = max([len(x) for x in df_train + df_val + df_test])

enc_df_train, y_train = build_embeddings_df(v2, df_train, 100, max_len_sentence )
enc_df_val, y_val = build_embeddings_df(v3, df_val, 100, max_len_sentence)
enc_df_test, y_test = build_embeddings_df(v4, df_test, 100, max_len_sentence)







/usr/local/lib/python3.7/dist-packages/ipykernel_launcher.py:21: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.


In [12]:

print(enc_df_train.shape)
print(enc_df_val.shape)
print(enc_df_test.shape)

(1963, 249, 100)
(1299, 249, 100)
(652, 249, 100)


In [13]:
#!pip install tensorflow
from sklearn.preprocessing import LabelBinarizer

def onehot(y ,label_binarizer, max_l, is_fit = False):

  
  y_onehot = []
  if is_fit:
    label_binarizer.fit([word for sentence in y for word in sentence])

  for sentence in y:
    y_onehot_row = label_binarizer.transform(sentence)
    padding_row = np.zeros((max_l - len(y_onehot_row), len(label_binarizer.classes_)))
    y_onehot.append(
        np.vstack(
            ( 
               y_onehot_row,
               padding_row,  
            )
        )
    )
  return np.array(y_onehot)

label_binarizer = LabelBinarizer()
enc_y_train = onehot(y_train, label_binarizer, max_len_sentence, is_fit = True)
enc_y_val = onehot(y_val, label_binarizer, max_len_sentence)
enc_y_test = onehot(y_test, label_binarizer, max_len_sentence)

print(enc_y_train.shape)
print(enc_y_val.shape)
print(enc_y_test.shape)

(1963, 249, 45)
(1299, 249, 45)
(652, 249, 45)


In [14]:
from keras import Sequential, Input
from keras.layers import Dense, Bidirectional, LSTM, Activation, TimeDistributed, Embedding

model = Sequential()
#model.add(Embedding(input_dim = (len(v4.vocab), 50)))
model.add(Bidirectional(LSTM(64, return_sequences = True), input_shape = (249, 100)))
model.add(Bidirectional(LSTM(128, return_sequences = True), input_shape = (128, 100)))
model.add(TimeDistributed(Dense(45)))
model.add(Activation('softmax'))
model.compile(loss='categorical_crossentropy', optimizer='rmsprop', metrics=["accuracy"])



In [15]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 bidirectional (Bidirectiona  (None, 249, 128)         84480     
 l)                                                              
                                                                 
 bidirectional_1 (Bidirectio  (None, 249, 256)         263168    
 nal)                                                            
                                                                 
 time_distributed (TimeDistr  (None, 249, 45)          11565     
 ibuted)                                                         
                                                                 
 activation (Activation)     (None, 249, 45)           0         
                                                                 
Total params: 359,213
Trainable params: 359,213
Non-trainable params: 0
__________________________________________________

In [16]:
history = model.fit(enc_df_train, enc_y_train, batch_size=4, epochs=30, validation_data = (enc_df_val, enc_y_val))


Epoch 1/30
491/491 [==============================] - 33s 50ms/step - loss: 0.2060 - accuracy: 0.0440 - val_loss: 0.1158 - val_accuracy: 0.0692
Epoch 2/30
491/491 [==============================] - 23s 47ms/step - loss: 0.0837 - accuracy: 0.0779 - val_loss: 0.0707 - val_accuracy: 0.0795
Epoch 3/30
491/491 [==============================] - 25s 51ms/step - loss: 0.0579 - accuracy: 0.0836 - val_loss: 0.0568 - val_accuracy: 0.0831
Epoch 4/30
491/491 [==============================] - 24s 48ms/step - loss: 0.0470 - accuracy: 0.0861 - val_loss: 0.0500 - val_accuracy: 0.0847
Epoch 5/30
491/491 [==============================] - 23s 47ms/step - loss: 0.0403 - accuracy: 0.0876 - val_loss: 0.0457 - val_accuracy: 0.0853
Epoch 6/30
491/491 [==============================] - 24s 48ms/step - loss: 0.0354 - accuracy: 0.0888 - val_loss: 0.0420 - val_accuracy: 0.0865
Epoch 7/30
491/491 [==============================] - 23s 47ms/step - loss: 0.0315 - accuracy: 0.0898 - val_loss: 0.0387 - val_accuracy:

KeyboardInterrupt: ignored